# 03c · 출생-사망 과정과 정상분포 (Birth–Death Processes and Stationary Distributions)

| 항목 | 내용 |
|---|---|
| 교재 지도 | Durrett 3e §4.3 (극한 거동, $\pi Q=0$, 세부 균형), §4.5 도입부 (M/M/1, M/M/∞) · Ross 12e §6.3 (출생-사망 과정), §6.5 (극한 확률, 균형 방정식, 선형 성장+이입) · Norris §3.5 (불변분포: $\pi Q=0\iff\pi P(t)=\pi$), §3.8 (에르고딕 정리) · Lawler 2e §3.3 (출생-사망 과정의 곱 형태와 존재 조건) |
| 선수 노트북 | 03b (콜모고로프 방정식, $P(t)=e^{Qt}$), 01d (DTMC 정상분포) |
| 예상 소요 | 90분 |
| 상태 | draft |

이 노트북의 단 하나의 아이디어: **정상분포는 "들어오는 흐름 = 나가는 흐름"($\pi Q=0$)이고, 출생-사망 과정에서는 상태 $n$과 $n+1$ 사이에 벽을 세워 세면 그 방정식이 $\lambda_n\pi_n=\mu_{n+1}\pi_{n+1}$로 풀려 곱 형태가 나온다.** 곱 형태의 세 얼굴(기하·푸아송·음이항)을 닫힌 형식으로 쓰고, 에르고딕 정리(시간평균 → $\pi$)와 과도 분포(M/M/∞의 푸아송)를 시뮬레이션으로 확인한다.

## 0. 준비 (Setup)

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Markdown
from scipy.stats import poisson, nbinom
from spkit import rng_for, fast, scale, setup_plots
from spkit.mc import mc_estimate, mc_proportion, compare_table, assert_close
from spkit.ctmc import (is_generator, birth_death_generator, birth_death_stationary,
                        stationary_ctmc, mm1_stationary, embedded_chain, gillespie,
                        time_average_occupation)
from spkit.markov import stationary, tv_distance
from spkit.plots import plot_paths, plot_hist_vs_pmf

SEED, rng = rng_for("03c")
setup_plots()
N_PATHS = scale(20_000)   # FAST(SPKIT_FAST=1) 모드에서는 1/10 크기
print(f"SEED={SEED}, fast={fast()}, N_PATHS={N_PATHS}")

## 1. Recall — 지난 노트북에서

책을 덮고 먼저 답을 적어 본 뒤 정답을 펼쳐 보세요.

**Q1.** (03b) 콜모고로프 전진·후진 방정식은 각각 무엇이고, 유한 상태에서 해는?

<details><summary>정답</summary>

전진 $P'(t)=P(t)Q$, 후진 $P'(t)=QP(t)$. $P(0)=I$인 유일한 해는 $P(t)=e^{Qt}=\sum_n (Qt)^n/n!$이다. 특히 $P'(0)=Q$.

</details>

**Q2.** (03b) 기약 유한 CTMC에서 $P_{ij}(t)$는 $t\to\infty$에 무엇으로 가는가? DTMC와 달리 필요 없는 조건은?

<details><summary>정답</summary>

$\pi_j$(정상분포)로 수렴한다. 비주기성 조건이 필요 없다 — $t>0$이면 $P(t)$의 원소가 모두 양수라서 주기 현상이 생길 수 없다.

</details>

**Q3.** (03b) $P(t)\to\mathbf 1\pi$의 속도를 결정하는 양은? 복소 고윳값이면 어떤 현상이 생기나?

<details><summary>정답</summary>

$Q$의 0이 아닌 고윳값 중 실수부가 가장 큰 $\mathrm{Re}\,\lambda_2$ (스펙트럼 갭). 복소 고윳값이면 $\log\mathrm{TV}$ 곡선이 감쇠하면서 진동한다.

</details>

**Q4 (연결 고리).** (03a) 점프 사슬 $\Pi$와 생성원 $Q$의 관계, 그리고 $\Pi$의 대각원소는?

<details><summary>정답</summary>

$\Pi_{ij}=q_{ij}/q_i$ ($i\ne j$), $\Pi_{ii}=0$ (자기 전이 없음). 오늘의 함정 하나: $\Pi$의 정상분포 $\nu$는 CTMC의 정상분포 $\pi$가 **아니다** — 체류시간 $1/q_i$로 가중해야 한다 ($\pi_i\propto\nu_i/q_i$).

</details>

## 2. 이론 — 내 말로 다시 쓰기

### 2.1 정의

- **정상분포(stationary distribution).** $\pi\ge0$, $\sum_i\pi_i=1$, $\pi Q=0$. 성분별로 쓰면 **전역 균형(global balance)**
  $$\sum_{i\ne j}\pi_i q_{ij}=\pi_j q_j\qquad(\text{상태 }j\text{로 들어오는 흐름} = j\text{에서 나가는 흐름}).$$
  (Durrett 3e §4.3; Ross 12e §6.5의 '비율 균형 방정식'.) spkit: `stationary_ctmc(Q)`는 $\pi Q=0$의 한 식을 $\sum\pi=1$로 바꿔 선형계로 푼다.
- **출생-사망 과정(birth–death process).** 상태 $\{0,1,\dots\}$ 또는 $\{0,\dots,K\}$, $n\to n+1$ 비율 $\lambda_n$(출생), $n\to n-1$ 비율 $\mu_n$(사망, $\mu_0=0$). 이웃으로만 움직이므로 $Q$는 **삼중대각**이다. (Ross 12e §6.3; Lawler 2e §3.3.) spkit: `birth_death_generator(birth, death)` — `birth[i]`는 $i\to i+1$ ($i=0..n-1$, 길이 $n$), `death[i]`는 $i\to i-1$ ($i=1..n$, 길이 $n+1$, `death[0]`은 무시), 반환은 $(n+1)\times(n+1)$.
- **절단(cut) 방정식.** $\lambda_n\pi_n=\mu_{n+1}\pi_{n+1}$ — 상태 $n$과 $n+1$ 사이에 벽을 세우면 오른쪽으로 넘는 흐름과 왼쪽으로 넘는 흐름이 같다.
- **세부 균형(detailed balance).** $\pi_iq_{ij}=\pi_jq_{ji}$ 모든 $i,j$. 성립하면 사슬은 **가역(reversible)** (01f의 CTMC 판). 출생-사망 과정은 항상 세부 균형을 만족한다 (그래프가 선분이라 사이클이 없다).
- **곱 형태(product form).**
  $$\pi_n=\pi_0\prod_{k=0}^{n-1}\frac{\lambda_k}{\mu_{k+1}},\qquad \pi_0=\Big(\sum_n\prod_{k<n}\lambda_k/\mu_{k+1}\Big)^{-1}.$$
  무한 상태에서는 합이 유한해야 정상분포가 존재한다. spkit `birth_death_stationary(birth, death)`는 로그 누적합 $\sum\log(\lambda_k/\mu_{k+1})$으로 계산해 큰 상태 수에서도 오버플로가 없다.
- **세 예 (이 노트북의 수치).**
  (i) **M/M/1/K**: $\lambda_n=\lambda\ (n<K)$, $\mu_n=\mu$ → $\pi_n=\dfrac{(1-\rho)\rho^n}{1-\rho^{K+1}}$, $\rho=\lambda/\mu$ (`mm1_stationary(lam, mu, K)`와 같음). 수치: $\lambda=1,\mu=1.5,K=10$.
  (ii) **M/M/∞**: $\lambda_n=\lambda$, $\mu_n=n\mu$ → $\pi=\mathrm{Poisson}(\lambda/\mu)$. 수치: $\lambda=3,\mu=1$ (계산은 $n_{\max}=30$으로 절단).
  (iii) **선형 성장 + 이입(linear growth with immigration)**: $\lambda_n=n\lambda+\theta$, $\mu_n=n\mu$ ($\lambda<\mu$) → $\pi=\mathrm{NegBin}(r=\theta/\lambda,\ p=\lambda/\mu)$, $\pi_n=\binom{n+r-1}{n}(1-p)^rp^n$, 평균 $\theta/(\mu-\lambda)$. 수치: $\lambda=0.5,\mu=1,\theta=1$ → $r=2,p=1/2$, $\pi_n=(n+1)/2^{n+2}$ ($n_{\max}=60$).
- **시간평균 점유율(time-average occupation).** $\frac1T\int_0^T\mathbf 1\{X_s=i\}\,ds$ (spkit `time_average_occupation(times, states, T, n_states)`). 에르고딕 정리는 이것이 $\pi_i$로 간다는 주장이다.
- **점프 사슬의 정상분포 $\nu$** ($\nu\Pi=\nu$)와의 관계: $\pi_i\propto\nu_i/q_i$ — 방문 횟수 비율 × 평균 체류시간.

### 2.2 정리 1 — $\pi Q=0\iff\pi P(t)=\pi$ (Norris §3.5, Durrett 3e §4.3)

> 유한 상태 CTMC와 확률벡터 $\pi$에 대해 $\pi Q=0$ ⇔ 모든 $t\ge0$에서 $\pi P(t)=\pi$. 즉 $X_0\sim\pi$이면 모든 $t$에 $X_t\sim\pi$ (과정이 **정상(stationary)**).

**가정이 왜 필요한가**
- **유한 상태.** $P(t)=\sum_n(Qt)^n/n!$ 급수와 $t=0$에서의 항별 미분을 정당화한다. 무한 상태에서는 $\pi Q=0$의 해가 정규화되지 않거나(M/M/1, $\rho\ge1$), 폭발 사슬에서는 동치 자체가 깨질 수 있다.
- **$\pi\ge0,\ \sum\pi=1$.** $\pi Q=0$은 동차 방정식이라 스칼라배의 자유도가 있다; 정규화가 그중 분포를 고른다.

<details><summary>증명 (완전 증명)</summary>

(⇒) $\pi Q=0$이면 귀납으로 $\pi Q^n=(\pi Q)Q^{n-1}=0$ ($n\ge1$). 따라서
$$\pi P(t)=\pi\sum_{n\ge0}\frac{t^n}{n!}Q^n=\pi I+\sum_{n\ge1}\frac{t^n}{n!}\,\pi Q^n=\pi.$$

(⇐) 모든 $t$에서 $\pi P(t)=\pi$이면 양변을 $t$로 미분할 수 있고(유한 행렬 지수는 해석적), $t=0$에서 $\pi P'(0)=0$. 후진(또는 전진) 방정식에서 $P'(0)=Q$이므로 $\pi Q=0$.

(유일성, 기약일 때) 03b 정리 3에 의해 $P(t)\to\mathbf 1\pi$. $\pi'P(t)=\pi'$인 다른 확률벡터 $\pi'$가 있다면 $t\to\infty$에서 $\pi'=\pi'\mathbf 1\pi=\pi$. $\square$

</details>

### 2.3 정리 2 — 출생-사망 곱 형태 (Lawler 2e §3.3, Ross 12e §6.5)

> $\lambda_n>0\ (n<K)$, $\mu_n>0\ (1\le n\le K)$인 출생-사망 과정의 정상분포는
> $$\pi_n=\pi_0\prod_{k=0}^{n-1}\frac{\lambda_k}{\mu_{k+1}},$$
> 이며 $K=\infty$일 때는 $\sum_n\prod_{k<n}\lambda_k/\mu_{k+1}<\infty$일 때만 존재한다. 더불어 세부 균형 $\pi_n\lambda_n=\pi_{n+1}\mu_{n+1}$이 성립한다 (가역).

**가정이 왜 필요한가**
- **$\lambda_n,\mu_{n+1}>0$ (기약).** 어느 비율이 0이면 상태공간이 끊겨 정상분포가 유일하지 않거나 흡수가 생긴다.
- **합의 유한성.** 정규화 상수 $\pi_0^{-1}$이 발산하면 $\pi Q=0$의 해는 있어도 분포가 아니다 (M/M/1에서 $\rho\ge1$이면 $\sum\rho^n=\infty$).
- **삼중대각 (이웃으로만 이동).** 절단 논법은 '벽을 넘는 경로가 한 종류'라는 데 기댄다. 일반 사슬에서는 절단 방정식이 성립하지 않고 전역 균형만 남는다.

<details><summary>증명 (완전 증명)</summary>

$\pi Q=0$을 성분별로 쓴다. $Q$의 $n$번째 열에는 $q_{n-1,n}=\lambda_{n-1}$, $q_{nn}=-(\lambda_n+\mu_n)$, $q_{n+1,n}=\mu_{n+1}$만 있으므로

- $n=0$: $-\lambda_0\pi_0+\mu_1\pi_1=0$, 즉 $\lambda_0\pi_0=\mu_1\pi_1$.
- $n\ge1$: $\lambda_{n-1}\pi_{n-1}-(\lambda_n+\mu_n)\pi_n+\mu_{n+1}\pi_{n+1}=0$.

귀납 가정 $\lambda_{n-1}\pi_{n-1}=\mu_n\pi_n$을 두 번째 식에 대입하면 $\mu_n\pi_n-(\lambda_n+\mu_n)\pi_n+\mu_{n+1}\pi_{n+1}=0$, 즉 **절단 방정식** $\lambda_n\pi_n=\mu_{n+1}\pi_{n+1}$이 모든 $n$에서 성립한다. 이를 되풀이해 풀면 $\pi_{n}=\pi_{n-1}\lambda_{n-1}/\mu_n=\dots=\pi_0\prod_{k<n}\lambda_k/\mu_{k+1}$. $\sum\pi_n=1$로 $\pi_0$이 정해지며, 그 합이 발산하면 확률벡터가 없다. 세부 균형은 절단 방정식 그 자체다: $q_{n,n+1}=\lambda_n$, $q_{n+1,n}=\mu_{n+1}$이고 나머지 $q_{ij}=0$이므로 $\pi_iq_{ij}=\pi_jq_{ji}$가 모든 쌍에서 성립한다.

**세 예.** (i) 비 $\lambda_k/\mu_{k+1}=\rho$가 일정 → $\pi_n\propto\rho^n$ (기하; 유한 $K$에서 정규화 $\sum_{n\le K}\rho^n=(1-\rho^{K+1})/(1-\rho)$). (ii) $\prod_{k<n}\lambda/((k+1)\mu)=(\lambda/\mu)^n/n!$ → 푸아송, $\pi_0=e^{-\lambda/\mu}$. (iii) $\prod_{k<n}\dfrac{k\lambda+\theta}{(k+1)\mu}=\left(\dfrac\lambda\mu\right)^n\dfrac{\prod_{k<n}(k+\theta/\lambda)}{n!}=p^n\binom{n+r-1}{n}$ ($r=\theta/\lambda$, $p=\lambda/\mu$) → 음이항, $\pi_0=(1-p)^r$, 평균 $rp/(1-p)=\theta/(\mu-\lambda)$. $\square$

</details>

### 2.4 정리 3 — 에르고딕 정리 (Norris §3.8)

> 기약 양재귀 CTMC(유한이면 자동)에서 임의의 초기 상태에 대해
> $$\frac1T\int_0^T\mathbf 1\{X_s=i\}\,ds\ \xrightarrow[T\to\infty]{}\ \pi_i\quad\text{a.s.},\qquad \pi_i=\frac{1}{q_i\,E_i[T_i]}$$
> ($T_i$는 $i$로의 재방문 시간).

**가정이 왜 필요한가**
- **기약.** 아니면 시간평균이 출발한 닫힌 성분에 의존한다.
- **양재귀.** 재방문 시간의 기댓값이 유한해야 재생-보상 논법의 비율 '사이클당 체류 / 사이클 길이'가 정의된다.

**증명은 인용** (Norris §3.8, 에르고딕 정리; 정리 번호는 책에서 확인). 아이디어만: $i$를 방문하는 시각들은 재생 과정을 이루고(04c), 한 사이클 동안 $i$에 머무는 시간의 기댓값은 $1/q_i$, 사이클 길이의 기댓값은 $E_i[T_i]$이므로 재생-보상 정리(04a)로 장기 비율이 $1/(q_iE_i[T_i])$이다. 이 노트북에서는 (a) 긴 경로 하나의 시간평균, (b) **정상 시작** 독립 경로들의 평균으로 수치 확인한다. (b)에서는 정리 1 덕분에 $E[\text{시간평균}]=\frac1T\int_0^TP(X_s=i)\,ds=\pi_i$가 **정확히** 성립하므로 편향이 0이고, 경로별 값이 i.i.d.라 SE가 정직하다.

### 2.5 정리 4 — M/M/∞의 과도 분포 (Ross 12e §6.3 예; 세분화는 02c)

> M/M/∞(도착 비율 $\lambda$, 각 손님이 독립 $\mathrm{Exp}(\mu)$ 서비스)가 빈 상태에서 출발하면
> $$X(t)\sim\mathrm{Poisson}\!\left(\frac\lambda\mu\left(1-e^{-\mu t}\right)\right),$$
> 이고 $t\to\infty$에서 $\mathrm{Poisson}(\lambda/\mu)=\pi$.

**가정이 왜 필요한가**
- **서버 무한 (서비스가 서로 독립).** 각 도착의 '아직 남아 있음'이 독립 베르누이가 되어 세분화가 적용된다. 서버가 유한하면 대기 때문에 독립성이 깨진다.
- **푸아송 도착.** 세분화 정리(02c)의 전제. 지수 서비스는 사실 필요 없다 — 일반 서비스 분포 $G$에서도 $\mathrm{Poisson}(\lambda\int_0^t\bar G(s)\,ds)$ (M/G/∞, 무감각성 예고).

<details><summary>증명 (완전 증명)</summary>

시각 $s\in(0,t)$에 도착한 손님이 시각 $t$에 아직 시스템에 있을 확률은 서비스가 $\mathrm{Exp}(\mu)$이므로 $p(s)=e^{-\mu(t-s)}$이고, 서로 다른 손님의 잔류 여부는 독립이다. 02c의 위치 의존 세분화(각 점을 위치 $s$에 따라 확률 $p(s)$로 남기면 남은 점들은 강도 $\lambda p(s)$의 푸아송 과정)에 의해 $[0,t]$의 도착 중 '남은' 손님 수는
$$\mathrm{Poisson}\Big(\int_0^t\lambda e^{-\mu(t-s)}\,ds\Big)=\mathrm{Poisson}\Big(\frac\lambda\mu(1-e^{-\mu t})\Big).$$

순서통계량 논법으로도 된다: $N(t)=n$이 주어지면 도착 시각은 $(0,t)$의 균등 순서통계량(02b)이고, 각 손님은 독립적으로 확률 $\bar p=\frac1t\int_0^te^{-\mu(t-s)}ds=\frac{1-e^{-\mu t}}{\mu t}$로 남는다. 따라서 $X(t)\mid N(t)=n\sim\mathrm{Bin}(n,\bar p)$이고 $N(t)\sim\mathrm{Poisson}(\lambda t)$의 이항 혼합은 $\mathrm{Poisson}(\lambda t\bar p)$ — 같은 모수다. $t\to\infty$에서 모수 → $\lambda/\mu$, 정리 2 (ii)의 $\pi$와 일치한다. $\square$

</details>

### 2.6 직관

절단 방정식은 **'벽 세기'** 다. 상태 $n$과 $n+1$ 사이에 벽을 세우면, 오른쪽으로 넘어간 횟수와 왼쪽으로 넘어온 횟수는 어느 시각에든 최대 1 차이이므로 장기 비율은 같다: 오른쪽으로 넘는 비율 $\lambda_n\pi_n$ = 왼쪽으로 넘는 비율 $\mu_{n+1}\pi_{n+1}$. 일반 사슬에서 이것이 안 되는 이유는 벽을 우회하는 경로(사이클)가 있기 때문이며, 그때는 전역 균형(들어오는 흐름 = 나가는 흐름)만 남는다. 곱 형태는 전역 균형의 특수해다.

M/M/∞가 푸아송인 것은 '푸아송 도착을 독립적으로 솎아내면 푸아송'(02c)과 같은 사실이다. 점프 사슬의 $\nu$와 $\pi$의 차이는 **방문 횟수**와 **머문 시간**의 차이 — 자주 가지만 잠깐 머무는 상태는 $\nu$에서 크고 $\pi$에서 작다.

### 2.7 함정 (traps)

- 점프 사슬 $\Pi$의 정상분포 $\nu$를 CTMC의 정상분포로 착각. 옳은 관계는 $\pi_i\propto\nu_i/q_i$ (4.2에서 표로 확인).
- 무한 상태공간에서 $\pi Q=0$의 해가 있어도 정규화가 안 되면 정상분포가 아니다 (M/M/1, $\rho\ge1$). **유한 절단은 항상 정상분포를 '만들어' 준다** — 절단 오차($\approx\rho^K$)를 확인해야 한다 (4.6, E3).
- `birth_death_generator(birth, death)`: `birth` 길이 $n$, `death` 길이 $n+1$, `death[0]`은 무시. 길이를 바꿔 넣으면 상태 수가 어긋난다.
- 시간평균의 SE를 i.i.d.처럼 `std/sqrt(n)`으로 계산하면 자기상관 때문에 크게 과소평가된다. 독립 경로 반복(경로별 시간평균의 표준편차, 4.4) 또는 배치 평균(03d)을 쓴다.
- 정상분포가 있다고 임의 시각 $t$의 분포가 $\pi$인 것은 아니다 — 초기 분포가 $\pi$일 때만. 0에서 출발한 M/M/∞의 $X(5)$는 $\mathrm{Poisson}(3(1-e^{-5}))\approx\mathrm{Poisson}(2.98)$이지 $\mathrm{Poisson}(3)$이 아니다.
- `gillespie`의 반환값에서 시각 $t$의 상태는 `states[np.searchsorted(times, t, side='right') - 1]`이다 (4.5의 `state_at`). spkit의 현재 구현은 $T$를 넘는 점프를 기록하지 않으므로 `states[-1]`이 $X_T$와 같지만, $t<T$인 격자 시각에서는 반드시 `searchsorted`를 써야 한다.

✍️ 내 말로: (책을 덮고 여기에 핵심 정의·정리를 다시 써 보세요 — (a) 왜 M/M/1에서 $\rho\ge1$이면 정상분포가 없는지 곱 형태로 설명하고, (b) '정상 시작 시뮬레이션의 시간평균은 편향이 없다'를 정리 1로 정당화해 보세요.)

## 3. Predict — 코드를 돌리기 전에 예측

아래 여섯 값을 손으로(또는 암산으로) 먼저 적어 보세요. 정답은 5절에서 코드가 계산합니다. 틀려도 괜찮습니다 — 예측과 결과의 차이가 학습 신호입니다.

In [ ]:
predictions = {
    # M/M/1/10 (lambda=1, mu=1.5)에서 시스템이 비어 있는 장기 시간 비율 pi_0은?
    "pi0_mm1k": None,
    # 같은 M/M/1/10의 장기 평균 손님 수 L = sum n pi_n 은? (절단 없는 M/M/1은 rho/(1-rho) = 2)
    "L_mm1k": None,
    # M/M/inf (lambda=3, mu=1)가 비어 있는 장기 비율은?
    "pi0_mminf": None,
    # 빈 M/M/inf (lambda=3, mu=1)에서 출발해 t=5일 때 시스템 안 손님 수의 기댓값은?
    "mean_x5_mminf": None,
    # 선형 성장+이입 (lambda=0.5, mu=1, theta=1)의 정상 평균 개체 수는?
    "mean_lgi": None,
    # 같은 과정에서 개체 수가 0인 장기 비율은?
    "pi0_lgi": None,
}

## 4. Simulate — 시뮬레이션

### 4.1 세 생성원과 곱 형태 — 정리 2

세 출생-사망 생성원을 `birth_death_generator`로 만들고, 곱 형태(`birth_death_stationary`) · 선형계 풀이(`stationary_ctmc`) · 닫힌 형식(기하/푸아송/음이항 pmf) 셋이 $10^{-10}$ 이내로 같은지 확인한다. M/M/∞와 선형 성장+이입은 무한 상태이므로 $n_{\max}$로 절단하고, 닫힌 형식도 같은 구간으로 재정규화해 비교한다 (절단 질량은 $10^{-15}$ 수준이라 실질적으로 0). scipy의 `nbinom.pmf(n, r, p_scipy)`는 성공 확률 $p_{\text{scipy}}=1-p$를 받으므로 $p_{\text{scipy}}=0.5$를 넣는다 (여기서는 $p=1/2$라 우연히 같다).

In [ ]:
# (i) M/M/1/10 : lambda=1, mu=1.5  (11 상태)
lam_k, mu_k, K = 1.0, 1.5, 10
b_k, d_k = np.full(K, lam_k), np.full(K + 1, mu_k)
# (ii) M/M/inf : lambda=3, mu=1, n_max=30
lam_i, mu_i, NMAX_I = 3.0, 1.0, 30
b_inf, d_inf = np.full(NMAX_I, lam_i), np.arange(NMAX_I + 1) * mu_i
# (iii) 선형 성장 + 이입 : lambda=0.5, mu=1, theta=1, n_max=60
lam_l, mu_l, th_l, NMAX_L = 0.5, 1.0, 1.0, 60
b_lgi, d_lgi = np.arange(NMAX_L) * lam_l + th_l, np.arange(NMAX_L + 1) * mu_l

Q_k, Q_inf, Q_lgi = (birth_death_generator(b, d) for b, d in [(b_k, d_k), (b_inf, d_inf), (b_lgi, d_lgi)])
assert is_generator(Q_k) and is_generator(Q_inf) and is_generator(Q_lgi)
pi_k, pi_inf, pi_lgi = (birth_death_stationary(b, d) for b, d in [(b_k, d_k), (b_inf, d_inf), (b_lgi, d_lgi)])
print("state counts:", Q_k.shape[0], Q_inf.shape[0], Q_lgi.shape[0], " row sums max|.|:", max(np.abs(Q.sum(1)).max() for Q in [Q_k, Q_inf, Q_lgi]))

곱 형태 · 선형계 풀이 · 닫힌 형식 세 가지를 $10^{-10}$ 이내로 맞춘다.

In [ ]:
# 닫힌 형식 (정리 2의 세 예)
rho = lam_k / mu_k
n_k, n_i, n_l = np.arange(K + 1), np.arange(NMAX_I + 1), np.arange(NMAX_L + 1)
geo_k = (1 - rho) * rho**n_k / (1 - rho**(K + 1))
pois_i = poisson.pmf(n_i, lam_i / mu_i) / poisson.cdf(NMAX_I, lam_i / mu_i)
r_l, p_l = th_l / lam_l, lam_l / mu_l
nb_l = nbinom.pmf(n_l, r_l, 1 - p_l) / nbinom.cdf(NMAX_L, r_l, 1 - p_l)

for name, Q, pi, closed, extra in [("M/M/1/10", Q_k, pi_k, geo_k, mm1_stationary(lam_k, mu_k, K)),
                                   ("M/M/inf", Q_inf, pi_inf, pois_i, None),
                                   ("LGI", Q_lgi, pi_lgi, nb_l, (n_l + 1) / 2.0**(n_l + 2))]:
    np.testing.assert_allclose(pi, stationary_ctmc(Q), atol=1e-10)   # 곱 형태 = 선형계 풀이
    np.testing.assert_allclose(pi, closed, atol=1e-10)                 # 곱 형태 = 닫힌 형식
    if extra is not None:
        np.testing.assert_allclose(pi, extra, atol=1e-10)
    print(f"{name:9s} states={Q.shape[0]:3d}  pi_0={pi[0]:.6f}  max|product - closed|={np.abs(pi - closed).max():.1e}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(11, 3.4))
for ax, n, pi, title in [(axes[0], n_k, pi_k, "M/M/1/10 (rho=2/3): truncated geometric"),
                         (axes[1], n_i[:16], pi_inf[:16], "M/M/inf (lambda/mu=3): Poisson(3)"),
                         (axes[2], n_l[:16], pi_lgi[:16], "growth+immigration: NegBin(2, 1/2)")]:
    ax.stem(n, pi, linefmt="C0-", markerfmt="C0o", basefmt=" ")
    ax.set_title(title, fontsize=9); ax.set_xlabel("n"); ax.set_ylabel("pi_n")
fig.suptitle("Stationary pmfs of three birth-death processes (product form)")
plt.tight_layout(); plt.show()

세 pmf는 각각 기하(단조 감소), 푸아송(봉우리 3 근처), 음이항 $(n+1)/2^{n+2}$(봉우리 $n=1$)이다. 모두 같은 곱 형태 공식 하나에서 나왔다 — 비율의 비 $\lambda_k/\mu_{k+1}$가 상수인지, $1/(k+1)$처럼 줄어드는지, $(k+2)/(2(k+1))$처럼 $1/2$로 수렴하는지의 차이다.

### 4.2 절단 · 세부 균형 · 전역 균형, 그리고 점프 사슬 $\nu$와의 관계 (트랩 1)

정확한 항등식 검사: 세 경우 모두 절단(=세부 균형) $\pi_n\lambda_n=\pi_{n+1}\mu_{n+1}$과 전역 균형 $\pi Q=0$이 $10^{-12}$ 이내로 성립해야 한다. 그 다음 M/M/1/10의 점프 사슬 $\Pi$의 정상분포 $\nu$(`stationary`)를 구해 $\nu_n/q_n$을 정규화하면 $\pi$가 되는지 표로 본다.

In [ ]:
for name, Q, pi, b, d in [("M/M/1/10", Q_k, pi_k, b_k, d_k), ("M/M/inf", Q_inf, pi_inf, b_inf, d_inf),
                          ("LGI", Q_lgi, pi_lgi, b_lgi, d_lgi)]:
    cut = np.abs(pi[:-1] * b - pi[1:] * d[1:]).max()      # 절단 = 세부 균형
    glob = np.abs(pi @ Q).max()                            # 전역 균형 pi Q = 0
    assert cut < 1e-12 and glob < 1e-12, (name, cut, glob)
    print(f"{name:9s} max|pi_n lambda_n - pi_(n+1) mu_(n+1)| = {cut:.1e},  max|pi Q| = {glob:.1e}")

Pi_k, q_k = embedded_chain(Q_k)
nu_k = stationary(Pi_k)                     # 점프 사슬의 정상분포 (방문 횟수 비율)
w = nu_k / q_k
np.testing.assert_allclose(w / w.sum(), pi_k, atol=1e-10)
lines = ["| n | pi_n (CTMC) | nu_n (jump chain) | q_n | nu_n/q_n normalized |", "|---|---|---|---|---|"]
lines += [f"| {n} | {pi_k[n]:.4f} | {nu_k[n]:.4f} | {q_k[n]:.1f} | {w[n] / w.sum():.4f} |" for n in range(K + 1)]
print(f"TV(nu, pi) = {tv_distance(nu_k, pi_k):.4f}  (nu != pi, but nu/q normalized == pi)")
Markdown("\n".join(lines))

$\nu$는 $\pi$와 다르다 (상태 0과 $K$는 이탈률 $q$가 작아 한 번 가면 오래 머무르므로 $\pi$에서 상대적으로 더 크다). 체류시간 $1/q_n$으로 가중한 뒤 정규화하면 정확히 $\pi$다.

### 4.3 긴 경로 하나의 시간평균 — 에르고딕 정리 (a)

M/M/1/10을 0에서 출발시켜 `gillespie`로 긴 경로 하나를 만들고, `time_average_occupation`으로 각 상태에 머문 시간 비율을 $\pi$와 비교한다. 경로 하나이므로 SE는 없다 — TV 거리만 본다 (trap 4: 자기상관 때문에 i.i.d. SE 공식은 쓸 수 없다).

In [ ]:
T_long = 200.0 if fast() else 2_000.0
times_l, states_l = gillespie(Q_k, 0, T_long, rng)
occ_long = time_average_occupation(times_l, states_l, T_long, K + 1)
print(f"T_long={T_long:g}: {len(times_l) - 1} jumps, TV(time average, pi) = {tv_distance(occ_long, pi_k):.4f}")

T_show = min(200.0, T_long)
m = times_l <= T_show
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
plot_paths(np.append(times_l[m], T_show), np.append(states_l[m], states_l[m][-1]), ax=axes[0], n_show=1, step=True, color="C0")
axes[0].set_title(f"M/M/1/10 sample path, first {T_show:g} time units"); axes[0].set_ylabel("customers in system")
axes[1].bar(n_k - 0.18, occ_long, 0.36, label=f"time average, one path, T={T_long:g}")
axes[1].bar(n_k + 0.18, pi_k, 0.36, alpha=0.6, label="stationary pi")
axes[1].set_title("Time-average occupation of one long path vs pi"); axes[1].set_xlabel("n"); axes[1].set_ylabel("fraction of time")
axes[1].legend(); plt.tight_layout(); plt.show()

경로가 길어질수록 막대가 $\pi$에 붙는다 (정리 3). 하지만 경로 하나로는 '얼마나 가까워야 정상인가'를 말할 수 없다 — 그래서 다음 절에서 독립 반복으로 SE를 만든다.

### 4.4 정상 시작 독립 반복 — 에르고딕 정리 (b), SE가 정직한 검증

각 경로의 시작 상태를 $\pi$에서 뽑으면(`rng.choice(K+1, p=pi_k)`) 정리 1에 의해 시간평균의 기댓값이 정확히 $\pi$이고, 경로들은 i.i.d.이므로 `mc_estimate`의 SE가 정직하다. 경로별로 (상태 0 점유율, 평균 손님 수 $\hat L=\sum_n n\cdot\mathrm{occ}_n$)을 모아 5절의 `pi0_mm1k`, `L_mm1k`와 비교한다.

In [ ]:
n4, T4 = scale(500), (20.0 if fast() else 100.0)
frac0, Lhat = np.empty(n4), np.empty(n4)
x0s = rng.choice(K + 1, size=n4, p=pi_k)             # 정상분포에서 시작
for i in range(n4):
    times, states = gillespie(Q_k, x0s[i], T4, rng)
    occ = time_average_occupation(times, states, T4, K + 1)
    frac0[i], Lhat[i] = occ[0], (n_k * occ).sum()
pi0_mc, L_mc = mc_estimate(frac0), mc_estimate(Lhat)
print(f"{n4} stationary-start paths x T={T4:g}:  fraction in 0 = {pi0_mc},  L-hat = {L_mc}")

fig, ax = plt.subplots()
ax.hist(frac0, bins=30, density=True, alpha=0.6, label=f"per-path time fraction in state 0 ({n4} paths, T={T4:g})")
ax.axvline(pi_k[0], color="C1", lw=2, label=f"pi_0 = {pi_k[0]:.4f}")
ax.axvline(pi0_mc.mean, color="C3", ls="--", label=f"mean of paths = {pi0_mc.mean:.4f}")
ax.set_title("Stationary-start M/M/1/10: time fraction in state 0 per path"); ax.set_xlabel("fraction of [0, T] in state 0"); ax.legend(fontsize=8)
plt.show()

경로별 점유율은 $\pi_0$ 주위에 넓게 퍼지지만(한 경로의 $T$가 짧아 분산이 큼) 평균은 편향 없이 $\pi_0$을 가리킨다. 이 산포의 표준편차$/\sqrt{n}$가 정직한 SE다.

### 4.5 M/M/∞의 과도 분포 — 정리 4

빈 M/M/∞($\lambda=3,\mu=1$)를 $t=5$까지 돌려 $X(5)$를 모은다. 시각 $t$의 상태는 `searchsorted`로 읽는다 (trap 6). 정리 4의 $\mathrm{Poisson}(3(1-e^{-5}))$ pmf와 히스토그램을 겹친다 — $\pi=\mathrm{Poisson}(3)$과는 모수가 $3e^{-5}\approx0.02$만큼 다르다.

In [ ]:
# helper (spkit에 없어서 여기서 정의)
def state_at(times, states, t):
    """gillespie 결과에서 시각 t의 상태 (마지막 점프 이후 상태 유지)."""
    return states[np.searchsorted(times, t, side="right") - 1]

n5, t5 = scale(2_000), 5.0
x5 = np.empty(n5, dtype=int)
for i in range(n5):
    times, states = gillespie(Q_inf, 0, t5, rng)
    x5[i] = state_at(times, states, t5)
m5 = (lam_i / mu_i) * (1 - np.exp(-mu_i * t5))       # 정리 4의 푸아송 모수
x5_mean, x5_p0 = mc_estimate(x5), mc_proportion(x5 == 0)
print(f"X(5) from empty M/M/inf: mean = {x5_mean},  P(X(5)=0) = {x5_p0};  Poisson parameter 3(1-e^-5) = {m5:.6f}")

ax = plot_hist_vs_pmf(x5, n_i[:13], poisson.pmf(n_i[:13], m5), label_pmf=f"Poisson({m5:.3f}) pmf (Theorem 4)")
ax.set_title(f"X(5) of M/M/inf started empty ({n5} paths) vs Poisson(3(1 - e^-5))"); ax.set_xlabel("customers at t = 5"); ax.set_ylabel("probability")
plt.show()

### 4.6 정상분포가 없는 경우 미리보기 (E3 연결)

M/M/1을 $K=50$으로 절단하고 $\rho\in\{2/3,\,1,\,1.2\}$로 `birth_death_stationary`를 계산한다. $\rho<1$이면 절단해도 무한 M/M/1의 기하분포와 거의 같지만, $\rho=1$이면 균등, $\rho>1$이면 질량이 $K$ 근처로 쏠린다 — 절단 없는 사슬에는 정상분포가 없는데 절단이 '만들어' 낸 것이다.

In [ ]:
K6 = 50
fig, ax = plt.subplots()
for rho6, c in [(2 / 3, "C0"), (1.0, "C1"), (1.2, "C3")]:
    pi6 = birth_death_stationary(np.full(K6, rho6), np.full(K6 + 1, 1.0))
    ax.plot(np.arange(K6 + 1), pi6, "o-", ms=3, color=c, label=f"rho = {rho6:.3g}: pi_0 = {pi6[0]:.2e}, pi_50 = {pi6[-1]:.2e}")
ax.set_yscale("log"); ax.set_xlabel("n"); ax.set_ylabel("pi_n (log scale)"); ax.legend(fontsize=8)
ax.set_title("Truncated M/M/1/50 stationary pmf: truncation manufactures a distribution")
plt.show()

로그 눈금에서 세 곡선은 기울기 $\log\rho$의 직선이다. $\rho=1.2$의 $\pi_0\approx1.8\times10^{-5}$는 '시스템이 거의 항상 꽉 차 있다'는 뜻이고, 이는 정리 2의 합 유한성이 깨진 신호다 (E3에서 절단 없이 돌려 본다).

## 5. Compare — 예측 · 시뮬레이션 · 닫힌 형식 비교

닫힌 형식 값을 모두 **공식에서** 계산한다 (스펙 숫자를 손으로 적지 않는다). MC 추정이 있는 네 행은 95% CI를 붙이고, 나머지 두 행은 곱 형태 수치(`birth_death_stationary`)와 닫힌 형식의 정확 비교다 (CI 없음).

In [ ]:
exact = {
    "pi0_mm1k": (1 - rho) / (1 - rho**(K + 1)),                   # 절단 기하의 정규화 상수
    "L_mm1k": float((n_k * geo_k).sum()),                          # sum n pi_n (닫힌 형식 pmf로 합산)
    "pi0_mminf": np.exp(-lam_i / mu_i),                            # Poisson(3) at 0
    "mean_x5_mminf": m5,                                           # 정리 4: (lambda/mu)(1 - e^{-mu t})
    "mean_lgi": th_l / (mu_l - lam_l),                             # NegBin 평균 r p/(1-p)
    "pi0_lgi": (1 - lam_l / mu_l) ** (th_l / lam_l),               # NegBin at 0: (1-p)^r
}
p0_x5_exact = np.exp(-m5)                                          # 추가 행: P(X(5)=0)
# 독립 확인: L 을 절단 기하의 닫힌 합 공식으로도 계산 (sum_{n<=K} n rho^n 의 유한 합)
L_closed = rho * (1 - (K + 1) * rho**K + K * rho**(K + 1)) / ((1 - rho) * (1 - rho**(K + 1)))
np.testing.assert_allclose(exact["L_mm1k"], L_closed, rtol=1e-12)
for k, v in exact.items():
    print(f"{k:14s} exact = {v:.6f}")
print(f"{'p0_x5_mminf':14s} exact = {p0_x5_exact:.6f}")

In [ ]:
rows = [
    {"name": "pi0_mm1k", "predicted": predictions["pi0_mm1k"], "estimate": pi0_mc, "exact": exact["pi0_mm1k"]},
    {"name": "L_mm1k", "predicted": predictions["L_mm1k"], "estimate": L_mc, "exact": exact["L_mm1k"]},
    {"name": "pi0_mminf (product form)", "predicted": predictions["pi0_mminf"], "estimate": float(pi_inf[0]), "exact": exact["pi0_mminf"]},
    {"name": "mean_x5_mminf", "predicted": predictions["mean_x5_mminf"], "estimate": x5_mean, "exact": exact["mean_x5_mminf"]},
    {"name": "p0_x5_mminf", "predicted": None, "estimate": x5_p0, "exact": p0_x5_exact},
    {"name": "mean_lgi (product form)", "predicted": predictions["mean_lgi"], "estimate": float((n_l * pi_lgi).sum()), "exact": exact["mean_lgi"]},
    {"name": "pi0_lgi (product form)", "predicted": predictions["pi0_lgi"], "estimate": float(pi_lgi[0]), "exact": exact["pi0_lgi"]},
]
Markdown(compare_table(rows))

각 행에서 불일치(`investigate`)가 나오면 뜻하는 바:
- **pi0_mm1k** — 정상 시작 시간평균이 $\pi_0$을 빗나간다: 시작 상태를 $\pi$에서 뽑지 않았거나(초기 편향), `time_average_occupation`의 시간 가중이 틀렸거나, $\pi$가 $\pi Q=0$의 해가 아니다.
- **L_mm1k** — $\hat L=\sum n\cdot\mathrm{occ}_n$의 평균이 $\sum n\pi_n$과 다르다: 상태 인덱스와 점유율의 정렬이 어긋났거나 절단 $K$가 생성원과 다르다.
- **pi0_mminf** — 곱 형태 수치가 $e^{-3}$과 다르다: $\mu_n=n\mu$ 대신 상수 $\mu$를 넣었거나 `death` 배열 길이가 어긋났다.
- **mean_x5_mminf** — $X(5)$의 평균이 $3(1-e^{-5})$과 다르다: 시각 5의 상태를 `states[-1]` 등으로 잘못 읽었거나 $\pi$의 평균 3과 혼동했다 (trap 5).
- **p0_x5_mminf** — 빈 확률이 $e^{-m_5}$와 다르다: $X(5)$가 푸아송이 아니다 — 서비스 독립성(서버 무한)이나 도착의 푸아송성이 깨졌다.
- **mean_lgi / pi0_lgi** — 곱 형태가 음이항과 다르다: $\lambda_n=n\lambda+\theta$의 이입 항 $\theta$를 빠뜨렸거나 scipy `nbinom`의 $p$ 매개변수화($1-p$)를 헷갈렸다.

In [ ]:
assert_close(pi0_mc, exact["pi0_mm1k"], k=4, name="pi0_mm1k")
assert_close(L_mc, exact["L_mm1k"], k=4, name="L_mm1k")
assert_close(x5_mean, exact["mean_x5_mminf"], k=4, name="mean_x5_mminf")
assert_close(x5_p0, p0_x5_exact, k=4, name="p0_x5_mminf")
np.testing.assert_allclose([pi_inf[0], (n_l * pi_lgi).sum(), pi_lgi[0]],
                           [exact["pi0_mminf"], exact["mean_lgi"], exact["pi0_lgi"]], atol=1e-10)
print("4 MC estimates within 4 SE of the closed forms; 3 product-form values match closed forms to 1e-10")

## 6. 연습문제

### E1 (계산)

손으로: 상태 $\{0,1,2\}$의 출생-사망 과정, $\lambda_0=1,\ \lambda_1=2,\ \mu_1=1,\ \mu_2=4$. 정상분포 $\pi$, 정상 평균 $\sum n\pi_n$, 점프 사슬 $\Pi$의 정상분포 $\nu$를 구하고 $\pi_i\propto\nu_i/q_i$를 확인하시오.

In [ ]:
# 여기에 풀이

<details><summary>정답</summary>

절단 방정식: $\pi_1=\pi_0\lambda_0/\mu_1=\pi_0$, $\pi_2=\pi_1\lambda_1/\mu_2=\pi_0/2$. 정규화 $\pi_0(1+1+\tfrac12)=1$ → $\pi=(0.4,\,0.4,\,0.2)$, 평균 $0\cdot0.4+1\cdot0.4+2\cdot0.2=0.8$.

이탈률 $q=(\lambda_0,\ \lambda_1+\mu_1,\ \mu_2)=(1,3,4)$, 점프 사슬
$$\Pi=\begin{pmatrix}0&1&0\\ 1/3&0&2/3\\ 0&1&0\end{pmatrix},\qquad \nu\Pi=\nu\ \Rightarrow\ \nu_0=\nu_1/3,\ \nu_2=2\nu_1/3\ \Rightarrow\ \nu\propto(1,3,2),\ \nu=(1/6,\,1/2,\,1/3).$$
$\nu_i/q_i=(1/6,\,1/6,\,1/12)\propto(2,2,1)\propto\pi$. 확인 완료 — $\nu$는 상태 1을 가장 자주 방문하지만($1/2$), 1의 체류가 짧아($1/q_1=1/3$) 시간 비율은 0과 같다.

```python
Q1 = birth_death_generator([1.0, 2.0], [0.0, 1.0, 4.0])
pi1 = birth_death_stationary([1.0, 2.0], [0.0, 1.0, 4.0])
Pi1, q1 = embedded_chain(Q1)
nu1 = stationary(Pi1)
print("pi =", pi1, " mean =", (np.arange(3) * pi1).sum())
print("q =", q1, " nu =", nu1, " nu/q normalized =", nu1 / q1 / (nu1 / q1).sum())
# pi = [0.4 0.4 0.2]  mean = 0.8 / q = [1. 3. 4.]  nu = [0.1667 0.5 0.3333]  nu/q normalized = [0.4 0.4 0.2]
```

</details>

### E2 (유도 후 시뮬레이션 검증)

선형 성장+이입($\lambda=0.5,\mu=1,\theta=1$)의 정상 평균이 $\theta/(\mu-\lambda)=2$임을 (a) 곱 형태(음이항 평균)로, (b) $\pi Q=0$을 쓰지 **않고** $\frac{d}{dt}E[X_t]=\theta+(\lambda-\mu)E[X_t]$를 유도해 정상 상태에서 얻으시오. 그런 다음 정상 시작(`rng.choice(61, p=pi_lgi)`) 질레스피 400경로, $T=50$(FAST: 200경로, $T=20$)의 경로별 시간평균 개체 수로 4 SE 이내 검증하시오.

In [ ]:
# 여기에 풀이

<details><summary>정답</summary>

(a) $\mathrm{NegBin}(r=2,p=1/2)$의 평균 $rp/(1-p)=2\cdot\frac{1/2}{1/2}=2$. 직접 합해도 $\sum_n n(n+1)/2^{n+2}=2$.

(b) 생성원으로 $f(x)=x$의 기댓값을 미분한다: $\frac{d}{dt}E[X_t]=E[(Qf)(X_t)]=E[\lambda_{X_t}\cdot(+1)+\mu_{X_t}\cdot(-1)]=E[\lambda X_t+\theta-\mu X_t]=\theta+(\lambda-\mu)E[X_t]$. 정상 상태에서는 좌변이 0이므로 $E[X]=\theta/(\mu-\lambda)=1/0.5=2$. (이 ODE는 $\lambda<\mu$일 때만 안정한 고정점을 가진다 — $\lambda\ge\mu$이면 평균이 발산, 정리 2의 조건과 같은 내용.)

```python
n2, T2 = scale(400), (20.0 if fast() else 50.0)
Lhat_lgi = np.empty(n2)
x0s = rng.choice(NMAX_L + 1, size=n2, p=pi_lgi)          # 정상 시작 -> 편향 0
for i in range(n2):
    times, states = gillespie(Q_lgi, x0s[i], T2, rng)
    occ = time_average_occupation(times, states, T2, NMAX_L + 1)
    Lhat_lgi[i] = (n_l * occ).sum()
est = mc_estimate(Lhat_lgi)
assert_close(est, th_l / (mu_l - lam_l), k=4, name="E2 stationary mean of LGI")
print(est, " exact =", th_l / (mu_l - lam_l))
# 예: 1.9799 ± 0.0554 (n=200)  exact = 2.0  (FAST)
```

</details>

### E3 (가정을 깨보기)

M/M/1을 $K=50$으로 절단하고 (a) $\rho=1.2$ ($\lambda=1.2,\mu=1$), (b) $\rho=1$로 `birth_death_stationary`를 계산해 $\pi_{50}$과 $\pi_0$을 보시오. 그리고 절단하지 않은 $\rho=1.2$ M/M/1을 `gillespie`($n_{\max}=2000$으로 충분히 큰 생성원)로 $T=200$까지 돌려 $X_T$의 성장을 관찰하시오. 무엇이 '정상분포'를 만들어냈는가?

In [ ]:
# 여기에 풀이

<details><summary>정답</summary>

(a) $\pi_n\propto1.2^n$이므로 질량이 $K$ 근처에 몰린다: $\pi_{50}=\dfrac{\rho^{50}(1-\rho)}{1-\rho^{51}}\to 1-1/\rho=1/6$에 가깝고($0.16668$), $\pi_0=\dfrac{1-\rho}{1-\rho^{51}}\approx1.8\times10^{-5}$. 이것은 **절단(유한 용량 $K$)이 만든 인공물**이다 — 용량이 있는 실제 M/M/1/K에서는 '거의 항상 꽉 참'이 맞는 답이지만, 무한 M/M/1의 극한 거동과는 무관하다.

(b) $\rho=1$이면 모든 비 $\lambda_k/\mu_{k+1}=1$이라 균등 $\pi_n=1/51\approx0.019608$.

절단 없는 $\rho=1.2$: 순 드리프트가 $\lambda-\mu=0.2$이므로 $X_T$는 $0.2T$ 근처로 **선형 발산**하며 되돌아오지 않는다 (대수의 법칙). 더 정확히는 E2 (b)와 같은 논법으로 $\frac{d}{dt}E[X_t]=\lambda-\mu\,P(X_t>0)=(\lambda-\mu)+\mu P(X_t=0)$이므로 $E[X_T]=(\lambda-\mu)T+\mu\int_0^TP(X_s=0)\,ds$; 0에서 보내는 총 시간의 기댓값은 (0 방문 횟수 기댓값 $\frac1{1-\mu/\lambda}=6$) × (체류 $1/\lambda$) $=\frac1{\lambda-\mu}=5$로 수렴하므로 $E[X_{200}]\approx40+5=45$ (`expm`으로 계산하면 $44.92$). 정리 2의 합 $\sum\rho^n$이 발산하므로 $\pi Q=0$의 해 $\pi_n\propto\rho^n$은 정규화할 수 없다 — 정상분포가 없다. 절단은 $\sum$을 강제로 유한하게 만들어 '정상분포'를 만들어낸 것이다.

```python
for rho3 in [1.2, 1.0]:
    pi3 = birth_death_stationary(np.full(50, rho3), np.full(51, 1.0))
    print(f"rho={rho3}: pi_50 = {pi3[50]:.6f}, pi_0 = {pi3[0]:.3e}")
# rho=1.2: pi_50 = 0.166682, pi_0 = 1.832e-05 / rho=1.0: pi_50 = 0.019608, pi_0 = 1.961e-02
Q_big = birth_death_generator(np.full(2000, 1.2), np.full(2001, 1.0))   # 절단 없는 것과 실질적으로 같음
T3, n3 = 200.0, 50
xT = np.empty(n3)
fig, ax = plt.subplots()
for i in range(n3):
    times, states = gillespie(Q_big, 0, T3, rng)
    xT[i] = state_at(times, states, T3)
    if i < 5:
        ax.step(np.append(times, T3), np.append(states, states[-1]), where="post", alpha=0.7)
ax.plot([0, T3], [0, (1.2 - 1.0) * T3], "k--", label="drift (lambda - mu) t = 0.2 t")
ax.set_title("Untruncated M/M/1 with rho = 1.2: no stationary distribution"); ax.set_xlabel("t"); ax.set_ylabel("X_t"); ax.legend()
plt.show()
print("X_200 over paths:", mc_estimate(xT), " (drift 0.2 * 200 = 40, plus time at 0: ~45)")
# 예: X_200 over paths: 46.2 ± 2.6 (n=50)  — 선형 성장, 정상분포 없음
```

</details>

## 7. 정리

**5줄 요약**
1. 정상분포는 $\pi Q=0$ (들어오는 흐름 = 나가는 흐름)이고, 이는 모든 $t$에 $\pi P(t)=\pi$와 동치다 (정리 1).
2. 출생-사망 과정은 절단 방정식 $\lambda_n\pi_n=\mu_{n+1}\pi_{n+1}$로 풀려 곱 형태 $\pi_n=\pi_0\prod_{k<n}\lambda_k/\mu_{k+1}$이 되고, 항상 가역이다 (정리 2).
3. M/M/1/K → 절단 기하, M/M/∞ → $\mathrm{Poisson}(\lambda/\mu)$, 선형 성장+이입 → $\mathrm{NegBin}(\theta/\lambda,\lambda/\mu)$; 무한 상태에서는 $\sum\prod\lambda_k/\mu_{k+1}<\infty$여야 존재한다.
4. 에르고딕 정리: 시간평균 점유율 → $\pi$ (정리 3); SE는 정상 시작 독립 경로 반복이나 배치 평균으로 정직하게 구한다.
5. 점프 사슬의 $\nu$와 CTMC의 $\pi$는 $\pi_i\propto\nu_i/q_i$로 다르고, 과도 분포(M/M/∞의 $\mathrm{Poisson}(\frac\lambda\mu(1-e^{-\mu t}))$, 정리 4)는 세분화로 얻는다.

**trap 카드**
- Q: 점프 사슬 $\Pi$의 정상분포 $\nu$를 구했다. 이것이 CTMC가 각 상태에 머무는 장기 시간 비율인가?
- A: 아니다. $\nu$는 '방문 횟수' 비율이고, 시간 비율은 체류시간 $1/q_i$로 가중해야 한다: $\pi_i=\dfrac{\nu_i/q_i}{\sum_j\nu_j/q_j}$. 예: E1에서 $\nu=(1/6,1/2,1/3)$이지만 $\pi=(0.4,0.4,0.2)$.

**다음 노트북: 03d (M/M/1 대기열, 리틀의 법칙, PASTA)** — 곱 형태의 가장 중요한 특수해 M/M/1($\pi_n=(1-\rho)\rho^n$)에서 $L,W$를 끌어내고, 분포와 무관한 회계 항등식 리틀의 법칙 $L=\lambda W$과 PASTA를 손님 단위 시뮬레이션으로 확인한다.

**log/progress.md 한 줄 템플릿**
`- [ ] 03c 출생-사망·정상분포 — 날짜: ____ / 예측 6개 중 맞은 것: __ / 막힌 곳: ____ / 다음: 03d`